# 02 - GraphRAG Retrieval Basics

This notebook demonstrates how GraphRAG retrieves context from the knowledge graph.

## Learning Objectives

1. Understand the difference between traditional RAG and GraphRAG
2. Use the SmartHomeRetriever for different retrieval strategies
3. See how retrieved context is formatted for LLM consumption
4. Understand when to use each retrieval strategy

## Prerequisites

- Completed notebook 01 (Neo4j exploration)
- Seed data loaded in Neo4j

## Setup

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from src.graph.connection import Neo4jConnection
from src.graph.retriever import SmartHomeRetriever, RetrievalStrategy

print("✓ Imports successful!")

✓ Imports successful!


In [2]:
# Create retriever
retriever = SmartHomeRetriever()

# Verify connection
stats = retriever.get_graph_stats()
print("Graph Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Graph Statistics:
  rooms: 5
  capabilities: 10
  devices: 13
  scenes: 5
  total_relationships: 68


## 1. Traditional RAG vs GraphRAG

### Traditional RAG (Document-based)
```
User Query → Embed Query → Vector Search → Top-K Chunks → LLM
```

**Example:**
- Query: "living room lights"
- Returns: Text chunks containing "living room" and "lights"
- Problem: May miss related context (capabilities, scenes)

### GraphRAG (Relationship-based)
```
User Query → Extract Entities → Graph Traversal → Subgraph → LLM
```

**Example:**
- Query: "living room lights"
- Returns: Living Room → All Devices → Their Capabilities → Related Scenes
- Benefit: Complete, structured context via relationship traversal

## 2. Retrieval Strategies

Our retriever supports multiple strategies:

In [3]:
# List available strategies
print("Available Retrieval Strategies:")
print("=" * 50)
for strategy in RetrievalStrategy:
    print(f"  - {strategy.value}")

Available Retrieval Strategies:
  - room_context
  - capability_search
  - scene_lookup
  - keyword_search
  - multi_room


## 3. Strategy 1: Room Context Retrieval

**Use when:** User mentions a specific room

**Retrieves:**
- Room details
- All devices in the room
- Device capabilities
- Applicable scenes
- Adjacent rooms

In [4]:
# Retrieve context for Living Room
result = retriever.get_room_context("living")

print(f"Strategy: {result.strategy.value}")
print(f"Query Params: {result.query_params}")
print(f"Metadata: {result.metadata}")
print("\n" + "=" * 50)
print("FORMATTED CONTEXT (what the LLM sees):")
print("=" * 50)
print(result.formatted_context)

Strategy: room_context
Query Params: {'room_name': 'living'}
Metadata: {'device_count': 6}

FORMATTED CONTEXT (what the LLM sees):
# Smart Home Context

## Room: Living Room
- Type: entertainment
- Floor: Ground Floor
- Adjacent to: Kitchen

### Devices (6 total)
- **Thermostat** (thermostat) ✓
  - Capabilities: temperature
- **Window Blinds** (blinds) ✓
  - Capabilities: open_close
- **Smart Speaker** (speaker) ✓
  - Capabilities: announce, play_music, volume, power
- **Floor Lamp** (light) ✓
  - Capabilities: dim, power
- **Ceiling Light** (light) ✓
  - Capabilities: color, dim, power
- **Smart TV** (television) ✓
  - Capabilities: input_select, volume, power

### Available Scenes
- **Party Mode**: Fun atmosphere for gatherings
  - Actions: Smart Speaker: play party playlist, high volume; Ceiling Light: colorful cycling
- **Movie Night**: Optimal settings for watching movies
  - Actions: Window Blinds: close; Floor Lamp: dim to 30%; Ceiling Light: dim to 20%, warm color; Smart TV: po

In [5]:
# Let's also see the raw results
import json

print("RAW RESULTS (structured data):")
print("=" * 50)
if result.raw_results:
    # Pretty print the first result
    print(json.dumps(result.raw_results[0], indent=2, default=str))

RAW RESULTS (structured data):
{
  "context": {
    "adjacent_rooms": [
      "Kitchen"
    ],
    "devices": [
      {
        "capabilities": [
          {
            "name": "temperature",
            "description": "Set target temperature",
            "parameters": [
              "target_celsius",
              "target_fahrenheit"
            ]
          }
        ],
        "device_id": "living_thermo_01",
        "name": "Thermostat",
        "type": "thermostat",
        "brand": "Nest",
        "status": "online"
      },
      {
        "capabilities": [
          {
            "name": "open_close",
            "description": "Open or close (blinds, curtains)",
            "parameters": [
              "position_percent"
            ]
          }
        ],
        "device_id": "living_blinds_01",
        "name": "Window Blinds",
        "type": "blinds",
        "brand": "Lutron",
        "status": "online"
      },
      {
        "capabilities": [
          {
           

### Teaching Point: Why Formatting Matters

The `formatted_context` is carefully structured for LLM consumption:

1. **Headers** (`#`, `##`, `###`) - Help LLM understand document structure
2. **Bullet points** - Clear enumeration of options
3. **Bold text** - Highlight important entity names
4. **Grouping** - Related information kept together

Raw JSON would work, but formatted text is:
- More token-efficient
- Easier for LLMs to parse
- Better for reasoning tasks

## 4. Strategy 2: Capability Search

**Use when:** User wants to do something (dim, play music, etc.)

**Retrieves:**
- All devices with the specified capability
- Which room each device is in
- Capability parameters

In [6]:
# Find all devices that can dim
result = retriever.get_devices_by_capability(["dim"])

print(f"Strategy: {result.strategy.value}")
print(f"Devices found: {result.metadata.get('device_count', 0)}")
print("\n" + "=" * 50)
print(result.formatted_context)

Strategy: capability_search
Devices found: 6

# Devices with capabilities: dim


## Home Office
- **Desk Lamp** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent

## Kitchen
- **Kitchen Light** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent

## Living Room
- **Floor Lamp** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent
- **Ceiling Light** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent

## Master Bedroom
- **Bedside Lamp** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent
- **Bedroom Light** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent


In [7]:
# Find devices that can play music
result = retriever.get_devices_by_capability(["play_music"])

print(f"Devices that can play music: {result.metadata.get('device_count', 0)}")
print("\n" + result.formatted_context)

Devices that can play music: 3

# Devices with capabilities: play_music


## Home Office
- **Office Speaker** (speaker)
  - play_music: Play music or audio content
    Parameters: source, playlist

## Living Room
- **Smart Speaker** (speaker)
  - play_music: Play music or audio content
    Parameters: source, playlist

## Master Bedroom
- **Bedroom Speaker** (speaker)
  - play_music: Play music or audio content
    Parameters: source, playlist


In [8]:
# Multiple capabilities - devices that can BOTH dim AND change color
result = retriever.get_devices_by_capability(["dim", "color"])

print("Devices with both 'dim' AND 'color' capabilities:")
print(result.formatted_context)

Devices with both 'dim' AND 'color' capabilities:
# Devices with capabilities: dim, color


## Home Office
- **Desk Lamp** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent

## Kitchen
- **Kitchen Light** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent

## Living Room
- **Floor Lamp** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent
- **Ceiling Light** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent
  - color: Change light color
    Parameters: color_hex, color_name

## Master Bedroom
- **Bedside Lamp** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent
  - color: Change light color
    Parameters: color_hex, color_name
- **Bedroom Light** (light)
  - dim: Adjust brightness level
    Parameters: brightness_percent
  - color: Change light color
    Parameters: color_hex, color_name


### Teaching Point: Intent-Based Retrieval

When user says "dim the lights", we need to:
1. Identify intent: `dim` capability
2. Find all devices with that capability
3. Provide LLM with options to choose from

This is different from keyword search - we're searching by **function**, not name.

## 5. Strategy 3: Scene Lookup

**Use when:** User mentions a mood or predefined scene

**Retrieves:**
- Scene details and description
- Which rooms it applies to
- Specific device actions

In [9]:
# Look up "Movie Night" scene
result = retriever.get_scene_context("movie")

print(f"Strategy: {result.strategy.value}")
print(f"Scenes found: {result.metadata.get('scenes_found', 0)}")
print("\n" + result.formatted_context)

Strategy: scene_lookup
Scenes found: 1

# Scene Details

## Movie Night
- Description: Optimal settings for watching movies
- Mood: relaxed
- Applies to: Living Room
- Typical actions: dim lights to 20%, close blinds, turn on TV, set warm color

### Device Actions:
- Window Blinds: close
- Floor Lamp: dim to 30%
- Ceiling Light: dim to 20%, warm color
- Smart TV: power on


In [10]:
# Look up bedtime scene
result = retriever.get_scene_context("bedtime")
print(result.formatted_context)

# Scene Details

## Bedtime
- Description: Relaxing settings for sleep preparation
- Mood: calm
- Applies to: Master Bedroom
- Typical actions: dim all lights, warm colors, play soft music, set cool temperature

### Device Actions:
- Bedside Lamp: warm orange, dim
- Bedroom Light: dim to 10%


### Teaching Point: Pre-defined vs Generated Actions

Scenes provide **pre-defined** action sets:
- Expert-curated configurations
- Guaranteed to work together
- Can be used as templates

The LLM can:
1. Use the scene directly
2. Modify it based on user preferences
3. Generate new actions inspired by the scene

## 6. Strategy 4: Keyword Search

**Use when:** User request is vague and needs disambiguation

**Retrieves:**
- Matching rooms
- Matching devices
- Matching scenes
- Matching capabilities

In [11]:
# Search for "cozy" - what entities match?
result = retriever.search_by_keywords(["cozy", "relax"])

print(f"Strategy: {result.strategy.value}")
print(f"Search terms: {result.metadata.get('search_terms', [])}")
print("\n" + result.formatted_context)

CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Aggregation column contains implicit grouping expressions. For example, in 'RETURN n.a, n.a + n.b + count(*)' the aggregation expression 'n.a + n.b + count(*)' includes the implicit grouping key 'n.b'. It may be possible to rewrite the query by extracting these grouping/aggregation expressions into a preceding WITH clause. Illegal expression(s): room_matches, device_matches, scene_matches (line 39, column 30 (offset: 1493))
"                rooms: [r IN room_matches WHERE r.name IS NOT NULL],"
                              ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

In [ ]:
# Search for "light" - should find devices and rooms
result = retriever.search_by_keywords(["light", "lamp"])
print(result.formatted_context)

### Teaching Point: Disambiguation

Keyword search helps when:
- User says "make it cozy" (what is "cozy"?)
- Multiple entities could match
- Need to clarify before taking action

The agent can then ask: "Did you mean the Movie Night scene?" or "Which room?"

## 7. Strategy 5: Multi-Room Context

**Use when:** Action spans multiple rooms

**Retrieves:**
- Multiple rooms' devices
- Combined capabilities

In [ ]:
# Get context for multiple rooms
result = retriever.get_multi_room_context(["living", "kitchen"])

print(f"Strategy: {result.strategy.value}")
print(f"Rooms found: {result.metadata.get('rooms_found', 0)}")
print("\n" + result.formatted_context)

## 8. Choosing the Right Strategy

Here's a decision tree for strategy selection:

```
User Request
    │
    ├─ Mentions specific room? → ROOM_CONTEXT
    │   "Turn on living room lights"
    │
    ├─ Mentions capability/action? → CAPABILITY_SEARCH
    │   "Dim all the lights"
    │
    ├─ Mentions scene/mood? → SCENE_LOOKUP
    │   "Movie night mode"
    │
    ├─ Mentions multiple rooms? → MULTI_ROOM
    │   "Prepare the house for guests"
    │
    └─ Vague/unclear? → KEYWORD_SEARCH
        "Make it cozy"
```

In [ ]:
# Let's simulate strategy selection for different requests

def analyze_request(request: str) -> str:
    """
    Simple heuristic for strategy selection.
    In a real system, an LLM would do this.
    """
    request_lower = request.lower()
    
    # Check for room mentions
    rooms = ["living room", "kitchen", "bedroom", "office", "bathroom"]
    mentioned_rooms = [r for r in rooms if r in request_lower]
    
    if len(mentioned_rooms) > 1:
        return "MULTI_ROOM"
    elif len(mentioned_rooms) == 1:
        return "ROOM_CONTEXT"
    
    # Check for scene/mood words
    scenes = ["movie", "party", "bedtime", "morning", "focus", "cozy", "relax"]
    if any(s in request_lower for s in scenes):
        return "SCENE_LOOKUP or KEYWORD_SEARCH"
    
    # Check for capability words
    capabilities = ["dim", "bright", "turn on", "turn off", "play", "music"]
    if any(c in request_lower for c in capabilities):
        return "CAPABILITY_SEARCH"
    
    return "KEYWORD_SEARCH"

# Test with different requests
requests = [
    "Turn on the living room TV",
    "Dim all the lights in the house",
    "Set up for movie night",
    "Make the bedroom and bathroom cozy",
    "Make it comfortable"
]

print("Strategy Selection Analysis")
print("=" * 60)
for req in requests:
    strategy = analyze_request(req)
    print(f"  \"{req}\"")
    print(f"  → {strategy}")
    print()

## 9. Complete Retrieval Example

Let's trace through a complete retrieval for a real user request.

In [ ]:
# User request: "Make the living room cozy for movie night"
user_request = "Make the living room cozy for movie night"
print(f"User Request: \"{user_request}\"")
print("=" * 60)

# Step 1: Get room context (primary)
print("\n📍 Step 1: Room Context Retrieval")
room_result = retriever.get_room_context("living")
print(f"   Found {room_result.metadata.get('device_count', 0)} devices")

# Step 2: Get scene context (secondary, for inspiration)
print("\n🎬 Step 2: Scene Context Retrieval")
scene_result = retriever.get_scene_context("movie")
print(f"   Found {scene_result.metadata.get('scenes_found', 0)} matching scenes")

# Step 3: Combine contexts for LLM
print("\n📝 Step 3: Combined Context for LLM")
combined_context = f"""
{room_result.formatted_context}

---

{scene_result.formatted_context}
"""

print(f"   Total context length: {len(combined_context)} characters")
print("\n" + "=" * 60)
print("COMBINED CONTEXT:")
print("=" * 60)
print(combined_context)

### What the LLM Receives

The LLM will receive a prompt like:

```
SYSTEM: You are a smart home assistant. Based on the following
available devices and scenes, create an action plan.

CONTEXT:
[Combined context from above]

USER: Make the living room cozy for movie night

ASSISTANT: Based on the available devices and the Movie Night scene,
here's my action plan:
1. Ceiling Light: dim to 20%, set warm color
2. Floor Lamp: dim to 30%
3. Window Blinds: close (0%)
4. Smart TV: power on
```

## 10. Exercises

In [ ]:
# Exercise 1: Get context for the bedroom
# What devices are available?



In [ ]:
# Exercise 2: Find all devices that can announce messages
# Hint: capability is 'announce'



In [ ]:
# Exercise 3: What happens when you search for "energy"?
# Are there any matching entities?



In [ ]:
# Exercise 4: Combine retrieval for "prepare the bedroom for sleep"
# What context would you retrieve?



## Summary

In this notebook, we learned:

1. **GraphRAG vs Traditional RAG**
   - Graph traversal vs vector similarity
   - Structured vs chunk-based retrieval

2. **Retrieval Strategies**
   - `ROOM_CONTEXT` - For room-specific requests
   - `CAPABILITY_SEARCH` - For action-based requests
   - `SCENE_LOOKUP` - For mood/scene requests
   - `KEYWORD_SEARCH` - For disambiguation
   - `MULTI_ROOM` - For whole-house requests

3. **Context Formatting**
   - Structured markdown for LLM consumption
   - Raw results for validation

4. **Strategy Selection**
   - Based on entities mentioned in user request
   - Multiple strategies can be combined

Next: `03_full_agent.ipynb` - Building the complete LangGraph agent

In [ ]:
# Note: Connection will be closed when the retriever goes out of scope
print("Notebook complete!")